# Star Schema and SQL
Build a star schema from the cleaned data


## Register cleaned data as tables
Load the cleaned Parquet into catalog tables so SQL can query it.

In [0]:
# read cleaned parquet and save as managed tables so sql can query them
spark.read.parquet("/Volumes/workspace/catenaretail/output/orders_cleaned") \
    .write.mode("overwrite").saveAsTable("workspace.catenaretail.orders_cleaned")

spark.read.parquet("/Volumes/workspace/catenaretail/output/customers_cleaned") \
    .write.mode("overwrite").saveAsTable("workspace.catenaretail.customers_cleaned")

print("tables created")

In [0]:
%sql
SELECT
  (SELECT COUNT(*) FROM workspace.catenaretail.orders_cleaned) AS orders,
  (SELECT COUNT(*) FROM workspace.catenaretail.customers_cleaned) AS customers;

## Build the dimensions
dim_customer, dim_product, and a date dimension with year, quarter, month, day.

In [0]:
%sql
-- dim_customer: one row per customer
CREATE OR REPLACE TABLE workspace.catenaretail.dim_customer (
  customer_id   STRING,
  customer_name STRING,
  region        STRING,
  signup_date   DATE
);

INSERT INTO workspace.catenaretail.dim_customer
SELECT customer_id, customer_name, region, signup_date
FROM workspace.catenaretail.customers_cleaned;

-- dim_product: distinct products seen in orders
CREATE OR REPLACE TABLE workspace.catenaretail.dim_product (
  product_id   STRING,
  product_name STRING
);

INSERT INTO workspace.catenaretail.dim_product
SELECT DISTINCT product_id, product_name
FROM workspace.catenaretail.orders_cleaned;

-- dim_date: one row per calendar date present in orders
CREATE OR REPLACE TABLE workspace.catenaretail.dim_date (
  date_key   DATE,
  year       INT,
  quarter    INT,
  month      INT,
  day        INT
);

INSERT INTO workspace.catenaretail.dim_date
SELECT DISTINCT
  order_date AS date_key,
  YEAR(order_date),
  QUARTER(order_date),
  MONTH(order_date),
  DAY(order_date)
FROM workspace.catenaretail.orders_cleaned;

## Build the fact table
One row per order, with measures and foreign keys to the dimensions.

In [0]:
%sql
-- fact_orders: one row per cleaned order, foreign keys to the dimensions
CREATE OR REPLACE TABLE workspace.catenaretail.fact_orders (
  order_id           STRING,
  customer_id        STRING,
  product_id         STRING,
  order_date         DATE,
  quantity           INT,
  amount             DECIMAL(12,2),
  is_orphan_customer BOOLEAN
);

INSERT INTO workspace.catenaretail.fact_orders
SELECT
  order_id,
  customer_id,
  product_id,
  order_date,
  quantity,
  amount,
  is_orphan_customer
FROM workspace.catenaretail.orders_cleaned;

SELECT COUNT(*) AS fact_rows FROM workspace.catenaretail.fact_orders;

## Query 1: monthly revenue by region

In [0]:
%sql
-- query 1: monthly revenue by region
-- revenue = quantity * amount is not needed here because amount is the line total,
-- so we sum amount directly. join fact to customer for region.
SELECT
  c.region,
  d.year,
  d.month,
  ROUND(SUM(f.amount), 2) AS revenue
FROM workspace.catenaretail.fact_orders f
JOIN workspace.catenaretail.dim_customer c ON f.customer_id = c.customer_id
JOIN workspace.catenaretail.dim_date d ON f.order_date = d.date_key
GROUP BY c.region, d.year, d.month
ORDER BY c.region, d.year, d.month;

## Query 2: top 5 products by revenue per quarter

In [0]:
%sql
-- query 2: top 5 products by revenue per quarter
-- rank products within each quarter, then keep the top 5 per quarter
WITH product_quarter AS (
  SELECT
    d.year,
    d.quarter,
    p.product_name,
    ROUND(SUM(f.amount), 2) AS revenue
  FROM workspace.catenaretail.fact_orders f
  JOIN workspace.catenaretail.dim_product p ON f.product_id = p.product_id
  JOIN workspace.catenaretail.dim_date d ON f.order_date = d.date_key
  GROUP BY d.year, d.quarter, p.product_name
),
ranked AS (
  SELECT *,
    ROW_NUMBER() OVER (PARTITION BY year, quarter ORDER BY revenue DESC) AS rn
  FROM product_quarter
)
SELECT year, quarter, product_name, revenue, rn AS rank
FROM ranked
WHERE rn <= 5
ORDER BY year, quarter, rank;

## Query 3: running total of spend per customer

In [0]:
%sql
-- query 3: running total of spend per customer over time, using a window function
-- for each customer, order their orders by date and accumulate spend
SELECT
  customer_id,
  order_date,
  order_id,
  amount,
  ROUND(SUM(amount) OVER (
    PARTITION BY customer_id
    ORDER BY order_date, order_id
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
  ), 2) AS running_total
FROM workspace.catenaretail.fact_orders
ORDER BY customer_id, order_date, order_id;

## Query 4: customers with no orders in the last 90 days

In [0]:
%sql
-- query 4: customers with no orders in the last 90 days
-- "last 90 days" is relative to the max order date in the data, not today
WITH bounds AS (
  SELECT MAX(order_date) AS max_date
  FROM workspace.catenaretail.fact_orders
),
recent_customers AS (
  SELECT DISTINCT f.customer_id
  FROM workspace.catenaretail.fact_orders f, bounds
  WHERE f.order_date >= DATE_SUB(bounds.max_date, 90)
)
SELECT
  c.customer_id,
  c.customer_name,
  c.region,
  MAX(f.order_date) AS last_order_date
FROM workspace.catenaretail.dim_customer c
LEFT JOIN workspace.catenaretail.fact_orders f ON c.customer_id = f.customer_id
WHERE c.customer_id NOT IN (SELECT customer_id FROM recent_customers)
GROUP BY c.customer_id, c.customer_name, c.region
ORDER BY last_order_date;